# Stage 1: Shortcut Candidate Extraction

**Objective:** Feature engineering to quantify suspected shortcut cues — hedging density, pronoun ratio, sentiment polarity, gender-proxy scores. Build baseline classifiers to test predictive power of shortcuts alone.

**Tools:** `LIWC`, `spaCy`, hedging lexicons, `scikit-learn` (LogisticRegression baselines)

In [2]:
import pandas as pd
import numpy as np
import spacy
from empath import Empath
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

/gpfs/automountdir/gpfs/homes/SEAS/home/g21775526/code/mh-shortcut-learning/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
nlp = spacy.load('en_core_web_sm')
lexicon = Empath()

cssrs = pd.read_csv('data/cssrs_standardized.csv')
mindset = pd.read_csv('data/mindset_standardized.csv')
umd = pd.read_csv('data/umd_standardized.csv')

cssrs['dataset'] = 'cssrs'
mindset['dataset'] = 'mindset'
umd['dataset'] = 'umd'

df = pd.concat([cssrs, mindset, umd], ignore_index=True)
df['text'] = df['sentence'].fillna('')
df['binary_label'] = df['label'].apply(lambda x: 1 if x in ['suicidal', 'at_risk', 'has_condition'] else 0)

print(f"Total samples: {len(df)}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nBinary label distribution:\n{df['binary_label'].value_counts()}")
print(f"\nDataset distribution:\n{df['dataset'].value_counts()}")

Total samples: 47237

Label distribution:
label
has_condition    16800
at_risk          15475
no_risk           5789
suicidal          5499
control           2400
supportive        1274
Name: count, dtype: int64

Binary label distribution:
binary_label
1    37774
0     9463
Name: count, dtype: int64

Dataset distribution:
dataset
umd        21264
mindset    19200
cssrs       6773
Name: count, dtype: int64


In [4]:
HEDGE_LEXICON = {
    'maybe', 'perhaps', 'possibly', 'probably', 'might', 'could', 'would', 'may',
    'seem', 'seems', 'seemed', 'seeming', 'appear', 'appears', 'appeared', 'appearing',
    'suggest', 'suggests', 'suggested', 'suggesting', 'think', 'thinks', 'thought',
    'feel', 'feels', 'felt', 'believe', 'believes', 'believed', 'guess', 'guessed',
    'suppose', 'supposes', 'supposed', 'assume', 'assumes', 'assumed', 'wonder',
    'kind of', 'sort of', 'somewhat', 'rather', 'quite', 'fairly', 'relatively',
    'approximately', 'roughly', 'around', 'about', 'nearly', 'almost', 'basically',
    'generally', 'usually', 'often', 'sometimes', 'occasionally', 'rarely', 'seldom',
    'tend', 'tends', 'tended', 'likely', 'unlikely', 'possible', 'impossible',
    'certain', 'uncertain', 'sure', 'unsure', 'doubt', 'doubts', 'doubted',
    'apparently', 'evidently', 'presumably', 'supposedly', 'allegedly', 'reportedly',
    'in my opinion', 'i think', 'i believe', 'i feel', 'i guess', 'i suppose',
    'it seems', 'it appears', 'to some extent', 'in a way', 'more or less'
}

FP_PRONOUNS = {'i', 'me', 'my', 'mine', 'myself', 'we', 'us', 'our', 'ours', 'ourselves'}
FP_SINGULAR = {'i', 'me', 'my', 'mine', 'myself'}

GENDER_FEMALE_MARKERS = {
    'she', 'her', 'hers', 'herself', 'girl', 'woman', 'mother', 'daughter', 'sister',
    'wife', 'girlfriend', 'female', 'ladies', 'lady', 'mom', 'mommy', 'aunt', 'niece'
}
GENDER_MALE_MARKERS = {
    'he', 'him', 'his', 'himself', 'boy', 'man', 'father', 'son', 'brother',
    'husband', 'boyfriend', 'male', 'guy', 'guys', 'dad', 'daddy', 'uncle', 'nephew'
}

In [5]:
def extract_features(text):
    doc = nlp(text.lower())
    tokens = [t.text for t in doc]
    n_tokens = max(len(tokens), 1)
    
    hedge_count = sum(1 for t in tokens if t in HEDGE_LEXICON)
    for phrase in ['kind of', 'sort of', 'i think', 'i believe', 'i feel', 'i guess', 
                   'i suppose', 'it seems', 'it appears', 'in my opinion', 'more or less']:
        hedge_count += text.lower().count(phrase)
    hedge_density = hedge_count / n_tokens
    
    fp_count = sum(1 for t in tokens if t in FP_PRONOUNS)
    fp_singular_count = sum(1 for t in tokens if t in FP_SINGULAR)
    fp_ratio = fp_count / n_tokens
    fp_singular_ratio = fp_singular_count / n_tokens
    
    female_count = sum(1 for t in tokens if t in GENDER_FEMALE_MARKERS)
    male_count = sum(1 for t in tokens if t in GENDER_MALE_MARKERS)
    gender_ratio = (female_count - male_count) / max(female_count + male_count, 1)
    
    empath_cats = lexicon.analyze(text, normalize=True) or {}
    negative_emotion = empath_cats.get('negative_emotion', 0)
    positive_emotion = empath_cats.get('positive_emotion', 0)
    sadness = empath_cats.get('sadness', 0)
    anxiety = empath_cats.get('nervousness', 0)
    health = empath_cats.get('health', 0)
    death = empath_cats.get('death', 0)
    
    n_sentences = max(len(list(doc.sents)), 1)
    avg_sent_len = n_tokens / n_sentences
    
    return {
        'hedge_density': hedge_density,
        'fp_ratio': fp_ratio,
        'fp_singular_ratio': fp_singular_ratio,
        'gender_ratio': gender_ratio,
        'negative_emotion': negative_emotion,
        'positive_emotion': positive_emotion,
        'sadness': sadness,
        'anxiety': anxiety,
        'health': health,
        'death': death,
        'avg_sent_len': avg_sent_len,
        'n_tokens': n_tokens
    }

In [ ]:
from tqdm import tqdm
tqdm.pandas()

features_list = df['text'].progress_apply(extract_features).tolist()
features_df = pd.DataFrame(features_list)
df = pd.concat([df.reset_index(drop=True), features_df], axis=1)

print(f"Extracted {len(features_df.columns)} features for {len(df)} samples")
features_df.describe()

 63%|██████▎   | 29906/47237 [09:13<04:28, 64.64it/s] 

In [ ]:
FEATURE_COLS = ['hedge_density', 'fp_ratio', 'fp_singular_ratio', 'gender_ratio',
                'negative_emotion', 'positive_emotion', 'sadness', 'anxiety', 
                'health', 'death', 'avg_sent_len']

def evaluate_single_feature(df, feature, target='binary_label', n_splits=5):
    X = df[[feature]].values
    y = df[target].values
    
    mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
    X, y = X[mask], y[mask]
    
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    clf = LogisticRegression(max_iter=1000, random_state=42)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')
    
    return {'feature': feature, 'mean_auc': aucs.mean(), 'std_auc': aucs.std()}

results = []
for feat in FEATURE_COLS:
    res = evaluate_single_feature(df, feat)
    results.append(res)
    
results_df = pd.DataFrame(results).sort_values('mean_auc', ascending=False)
results_df['is_shortcut'] = results_df['mean_auc'] > 0.62
results_df

In [ ]:
per_dataset_results = []
for dataset in df['dataset'].unique():
    df_sub = df[df['dataset'] == dataset]
    for feat in FEATURE_COLS:
        res = evaluate_single_feature(df_sub, feat)
        res['dataset'] = dataset
        per_dataset_results.append(res)

per_dataset_df = pd.DataFrame(per_dataset_results)
pivot = per_dataset_df.pivot(index='feature', columns='dataset', values='mean_auc').round(3)
pivot['is_shortcut'] = (pivot > 0.62).any(axis=1)
pivot.sort_values('is_shortcut', ascending=False)

In [ ]:
X_all = df[FEATURE_COLS].fillna(0).values
y_all = df['binary_label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

clf_all = LogisticRegression(max_iter=1000, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_feats_auc = cross_val_score(clf_all, X_scaled, y_all, cv=cv, scoring='roc_auc')

print(f"All shortcut features combined AUC: {all_feats_auc.mean():.3f} ± {all_feats_auc.std():.3f}")

In [ ]:
clf_all.fit(X_scaled, y_all)
coef_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'coefficient': clf_all.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print("Feature coefficients (all shortcuts combined):")
coef_df

In [ ]:
df.to_pickle('data/features_extracted.pkl')
results_df.to_csv('data/shortcut_candidate_aucs.csv', index=False)

shortcut_features = results_df[results_df['is_shortcut']]['feature'].tolist()
print(f"Candidate shortcuts (AUC > 0.62): {shortcut_features}")